# Sistemas Inteligentes I
## Búsqueda adversarial: algoritmo Minimax

**Autor:** Jairo I. Vélez B.  
**Desarrollado y resuelto por:** Juan David Ocampo González

---


# 1. Idea central: buscar cuando existe un adversario

En BFS, DFS o A* buscamos una ruta hacia un objetivo.  
En un juego ocurre algo diferente:

> después de que nosotros elegimos una acción, **otro jugador también elige**.

Por tanto, no basta con preguntarnos:

> ¿Cuál es la mejor jugada que puedo hacer?

También debemos considerar:

> ¿Qué hará mi oponente si intenta perjudicarme?

Minimax modela dos jugadores:

- **MAX:** intenta obtener el valor más alto posible.
- **MIN:** intenta obtener el valor más bajo posible.

Supondremos inicialmente que ambos jugadores actúan de manera racional.


# 2. Elementos de un problema adversarial

Un juego puede describirse mediante:

- **Estado:** configuración actual del juego.
- **Jugador actual:** indica quién debe mover.
- **Acciones:** movimientos legales desde el estado actual.
- **Resultado:** estado que se obtiene al aplicar una acción.
- **Estado terminal:** posición en la que la partida ha terminado.
- **Utilidad:** valor numérico asociado al resultado final.

Una convención sencilla puede ser:

| Resultado para MAX | Utilidad |
|---|---:|
| Victoria | `+1` |
| Empate | `0` |
| Derrota | `-1` |

En ejemplos más generales, la utilidad puede tomar cualquier valor numérico.


# 3. Primer árbol de juego

Consideremos el siguiente árbol.

- La raíz `A` pertenece a **MAX**.
- En el siguiente nivel juega **MIN**.
- Las hojas contienen valores de utilidad.

```text
                    A  (MAX)
              /         |         \
          B (MIN)    C (MIN)    D (MIN)
          / | \       / | \       / | \
         3  5  2     9  1  4     6  7  8
```

Pregunta:

> Si MIN juega racionalmente, ¿qué opción debería escoger MAX desde `A`?


In [1]:
arbol = {
    "A": ["B", "C", "D"],
    "B": ["B1", "B2", "B3"],
    "C": ["C1", "C2", "C3"],
    "D": ["D1", "D2", "D3"],
}

utilidades = {
    "B1": 3, "B2": 5, "B3": 2,
    "C1": 9, "C2": 1, "C3": 4,
    "D1": 6, "D2": 7, "D3": 8,
}

arbol, utilidades


({'A': ['B', 'C', 'D'],
  'B': ['B1', 'B2', 'B3'],
  'C': ['C1', 'C2', 'C3'],
  'D': ['D1', 'D2', 'D3']},
 {'B1': 3,
  'B2': 5,
  'B3': 2,
  'C1': 9,
  'C2': 1,
  'C3': 4,
  'D1': 6,
  'D2': 7,
  'D3': 8})

# 4. Razonamiento antes del algoritmo

Analicemos primero cada nodo de MIN.

### Nodo B

MIN puede elegir entre:

$$3,\ 5,\ 2$$

Por tanto:

$$\min(3,5,2)=2$$

### Nodo C

$$\min(9,1,4)=1$$

### Nodo D

$$\min(6,7,8)=6$$

La raíz pertenece a MAX, así que compara:

$$\max(2,1,6)=6$$

Por tanto, MAX debería elegir la rama `D`.

Esta idea es exactamente la que implementa **Minimax**.


# 5. Algoritmo Minimax

La definición recursiva puede escribirse como:

$$
V(s)=
\begin{cases}
U(s) & \text{si }s\text{ es terminal}\\
\max_{s' \in Sucesores(s)} V(s') & \text{si juega MAX}\\
\min_{s' \in Sucesores(s)} V(s') & \text{si juega MIN}
\end{cases}
$$

La recursión baja hasta los estados terminales y después los valores se
**propagan hacia arriba**.


In [2]:
def minimax(nodo, es_max, arbol, utilidades):
    # Caso base: nodo terminal
    if nodo in utilidades:
        return utilidades[nodo]

    valores = []

    for hijo in arbol[nodo]:
        valor = minimax(hijo, not es_max, arbol, utilidades)
        valores.append(valor)

    if es_max:
        return max(valores)
    else:
        return min(valores)


valor_raiz = minimax("A", True, arbol, utilidades)
valor_raiz


6

## 5.1 Obtener también la mejor jugada

Conocer el valor del estado es útil, pero normalmente necesitamos además saber:

> **¿qué acción debe ejecutar el jugador?**

La siguiente función devuelve el valor Minimax y el hijo seleccionado.


In [3]:
def mejor_jugada_minimax(nodo, es_max, arbol, utilidades):
    if nodo in utilidades:
        return utilidades[nodo], None

    opciones = []

    for hijo in arbol[nodo]:
        valor = minimax(hijo, not es_max, arbol, utilidades)
        opciones.append((valor, hijo))

    if es_max:
        valor, hijo = max(opciones, key=lambda x: x[0])
    else:
        valor, hijo = min(opciones, key=lambda x: x[0])

    return valor, hijo


valor, jugada = mejor_jugada_minimax("A", True, arbol, utilidades)

print("Valor Minimax:", valor)
print("Mejor jugada para MAX:", jugada)


Valor Minimax: 6
Mejor jugada para MAX: D


# 6. Minimax paso a paso

Para comprender mejor el algoritmo observaremos la recursión.

La sangría permite identificar la profundidad en el árbol.


In [4]:
def minimax_debug(nodo, es_max, arbol, utilidades, profundidad=0):
    sangria = "    " * profundidad
    jugador = "MAX" if es_max else "MIN"

    if nodo in utilidades:
        print(f"{sangria}{nodo}: terminal -> utilidad {utilidades[nodo]}")
        return utilidades[nodo]

    print(f"{sangria}{nodo}: turno de {jugador}")
    valores = []

    for hijo in arbol[nodo]:
        valor = minimax_debug(
            hijo,
            not es_max,
            arbol,
            utilidades,
            profundidad + 1
        )
        valores.append(valor)

    if es_max:
        resultado = max(valores)
    else:
        resultado = min(valores)

    print(f"{sangria}{nodo}: {jugador} selecciona {resultado}")
    return resultado


minimax_debug("A", True, arbol, utilidades)


A: turno de MAX
    B: turno de MIN
        B1: terminal -> utilidad 3
        B2: terminal -> utilidad 5
        B3: terminal -> utilidad 2
    B: MIN selecciona 2
    C: turno de MIN
        C1: terminal -> utilidad 9
        C2: terminal -> utilidad 1
        C3: terminal -> utilidad 4
    C: MIN selecciona 1
    D: turno de MIN
        D1: terminal -> utilidad 6
        D2: terminal -> utilidad 7
        D3: terminal -> utilidad 8
    D: MIN selecciona 6
A: MAX selecciona 6


6

### Respuestas a las Preguntas de análisis (Sección 6)

1. **¿Por qué MAX no selecciona directamente la hoja con valor `9`?**  
   Porque para llegar a esa hoja terminal ($C1 = 9$), MAX tendría que elegir primero la rama $C$. Sin embargo, en el nodo $C$ el turno de juego le pertenece a **MIN**. Dado que MIN juega de manera racional para perjudicar a MAX minimizando la utilidad, MIN elegirá la opción de menor valor de su rama, que es $C2 = 1$. Por lo tanto, si MAX comete el error de elegir $C$ esperando un 9, el adversario forzará un resultado final de 1.

2. **¿Qué supone Minimax sobre el comportamiento del adversario?**  
   Supone **racionalidad perfecta y juego adversarial estricto (juego de suma cero)**. Minimax asume que el oponente siempre elegirá la acción óptima para su propio beneficio (que equivale a la peor acción posible para MAX). No asume errores, descuidos ni aleatoriedad por parte del adversario.

3. **¿Qué ocurriría si MIN no escogiera siempre la opción de menor valor?**  
   Si MIN juega de manera subóptima (comete errores o juega al azar), MAX obtendría un resultado **mayor o igual** al valor predicho por Minimax. El valor Minimax representa una **cota inferior garantizada** del rendimiento de MAX frente a cualquier estrategia del oponente.

4. **¿Por qué los valores se calculan desde las hojas hacia la raíz?**  
   Por el principio de **inducción hacia atrás (backward induction)**. No es posible asignar un valor lógico a una decisión en el presente sin conocer previamente cuáles serán las consecuencias y respuestas finales en los estados terminales. Primero se evalúa el desenlace del juego en las hojas y luego se propagan los valores óptimos hacia arriba nivel por nivel.

5. **¿El valor de una hoja representa necesariamente una puntuación real del juego?**  
   No necesariamente. Representa una **función de utilidad ordinal o cardinal**, es decir, una medida cuantitativa diseñada por el programador para rankear qué tan favorable o desfavorable es un estado final (por ejemplo, $+1$ para victoria, $0$ para empate, $-1$ para derrota, o una evaluación heurística de ventaja posicional en ajedrez).


# 7. Un árbol con mayor profundidad

Ahora utilizaremos un árbol de tres decisiones.

```text
MAX → MIN → MAX → utilidad
```

Esto permite observar que los roles se alternan en cada nivel.


In [5]:
arbol_profundo = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["D1", "D2"],
    "E": ["E1", "E2"],
    "F": ["F1", "F2"],
    "G": ["G1", "G2"],
}

utilidades_profundo = {
    "D1": 3, "D2": 5,
    "E1": 6, "E2": 9,
    "F1": 1, "F2": 2,
    "G1": 0, "G2": -1,
}

minimax_debug("A", True, arbol_profundo, utilidades_profundo)


A: turno de MAX
    B: turno de MIN
        D: turno de MAX
            D1: terminal -> utilidad 3
            D2: terminal -> utilidad 5
        D: MAX selecciona 5
        E: turno de MAX
            E1: terminal -> utilidad 6
            E2: terminal -> utilidad 9
        E: MAX selecciona 9
    B: MIN selecciona 5
    C: turno de MIN
        F: turno de MAX
            F1: terminal -> utilidad 1
            F2: terminal -> utilidad 2
        F: MAX selecciona 2
        G: turno de MAX
            G1: terminal -> utilidad 0
            G2: terminal -> utilidad -1
        G: MAX selecciona 0
    C: MIN selecciona 0
A: MAX selecciona 5


5

## 7.1 Contar nodos evaluados

En árboles pequeños Minimax resulta sencillo.  
Sin embargo, el número de posiciones posibles puede crecer rápidamente.

Contaremos cuántos nodos visita el algoritmo.


In [6]:
def minimax_contando(nodo, es_max, arbol, utilidades, contador):
    contador["visitados"] += 1

    if nodo in utilidades:
        contador["terminales"] += 1
        return utilidades[nodo]

    valores = [
        minimax_contando(hijo, not es_max, arbol, utilidades, contador)
        for hijo in arbol[nodo]
    ]

    return max(valores) if es_max else min(valores)


contador = {"visitados": 0, "terminales": 0}
valor = minimax_contando(
    "A",
    True,
    arbol_profundo,
    utilidades_profundo,
    contador
)

print("Valor Minimax:", valor)
print("Nodos visitados:", contador["visitados"])
print("Hojas evaluadas:", contador["terminales"])


Valor Minimax: 5
Nodos visitados: 15
Hojas evaluadas: 8


# 8. Caso aplicado: juego de las piedras

Trabajaremos con un juego muy sencillo:

- Existe una pila con cierta cantidad de piedras.
- En cada turno un jugador puede retirar `1`, `2` o `3` piedras.
- El jugador que retira la **última piedra gana**.

Representaremos un estado como:

```python
(piedras_restantes, jugador)
```

donde:

- `jugador = 1` representa a MAX;
- `jugador = -1` representa a MIN.


In [7]:
MOVIMIENTOS = (1, 2, 3)

def movimientos_validos(piedras):
    return [m for m in MOVIMIENTOS if m <= piedras]


for n in range(1, 8):
    print(n, "piedras ->", movimientos_validos(n))


1 piedras -> [1]
2 piedras -> [1, 2]
3 piedras -> [1, 2, 3]
4 piedras -> [1, 2, 3]
5 piedras -> [1, 2, 3]
6 piedras -> [1, 2, 3]
7 piedras -> [1, 2, 3]


## 8.1 Utilidad del estado terminal

Cuando no quedan piedras, significa que el jugador anterior tomó la última.

Si el jugador que debe mover ahora es MAX, entonces MIN realizó la jugada anterior
y ganó. Por tanto, la utilidad para MAX es `-1`.

Si debe mover MIN, MAX realizó la jugada anterior y ganó. La utilidad es `+1`.


In [8]:
def minimax_piedras(piedras, turno_max):
    if piedras == 0:
        return -1 if turno_max else 1

    valores = []

    for retirar in movimientos_validos(piedras):
        valor = minimax_piedras(
            piedras - retirar,
            not turno_max
        )
        valores.append(valor)

    return max(valores) if turno_max else min(valores)


for piedras in range(1, 11):
    print(
        f"{piedras:2d} piedras -> valor Minimax:",
        minimax_piedras(piedras, True)
    )


 1 piedras -> valor Minimax: 1
 2 piedras -> valor Minimax: 1
 3 piedras -> valor Minimax: 1
 4 piedras -> valor Minimax: -1
 5 piedras -> valor Minimax: 1
 6 piedras -> valor Minimax: 1
 7 piedras -> valor Minimax: 1
 8 piedras -> valor Minimax: -1
 9 piedras -> valor Minimax: 1
10 piedras -> valor Minimax: 1


## 8.2 Encontrar la mejor jugada

Ahora determinaremos cuántas piedras debería retirar MAX.


In [9]:
def mejor_movimiento_piedras(piedras):
    opciones = []

    for retirar in movimientos_validos(piedras):
        valor = minimax_piedras(piedras - retirar, False)
        opciones.append((valor, retirar))

    mejor_valor, mejor_movimiento = max(opciones, key=lambda x: x[0])

    return {
        "retirar": mejor_movimiento,
        "valor": mejor_valor,
        "opciones": opciones,
    }


for piedras in range(1, 11):
    print(
        f"{piedras:2d} piedras ->",
        mejor_movimiento_piedras(piedras)
    )


 1 piedras -> {'retirar': 1, 'valor': 1, 'opciones': [(1, 1)]}
 2 piedras -> {'retirar': 2, 'valor': 1, 'opciones': [(-1, 1), (1, 2)]}
 3 piedras -> {'retirar': 3, 'valor': 1, 'opciones': [(-1, 1), (-1, 2), (1, 3)]}
 4 piedras -> {'retirar': 1, 'valor': -1, 'opciones': [(-1, 1), (-1, 2), (-1, 3)]}
 5 piedras -> {'retirar': 1, 'valor': 1, 'opciones': [(1, 1), (-1, 2), (-1, 3)]}
 6 piedras -> {'retirar': 2, 'valor': 1, 'opciones': [(-1, 1), (1, 2), (-1, 3)]}
 7 piedras -> {'retirar': 3, 'valor': 1, 'opciones': [(-1, 1), (-1, 2), (1, 3)]}
 8 piedras -> {'retirar': 1, 'valor': -1, 'opciones': [(-1, 1), (-1, 2), (-1, 3)]}
 9 piedras -> {'retirar': 1, 'valor': 1, 'opciones': [(1, 1), (-1, 2), (-1, 3)]}
10 piedras -> {'retirar': 2, 'valor': 1, 'opciones': [(-1, 1), (1, 2), (-1, 3)]}


### Respuestas a las Preguntas de análisis (Sección 8)

1. **¿Qué cantidades iniciales de piedras representan una posición desfavorable para MAX?**  
   Las cantidades iniciales que son **múltiplos exactos de 4**: $\{4, 8, 12, 16, \dots\}$. En cualquiera de estas cantidades, el valor Minimax para MAX es $-1$.

2. **¿Existe algún patrón?**  
   **Sí.** Existe una periodicidad modular estricta de período $k+1 = 3+1 = 4$:
   - Si $n \pmod 4 == 0$: la posición es perdedora (valor $-1$). Cualquier cantidad de piedras que retire el jugador de turno ($1, 2$ o $3$) dejará al oponente en una posición con residuo no nulo, permitiéndole al rival ganar.
   - Si $n \pmod 4 \neq 0$: la posición es ganadora (valor $+1$). La estrategia óptima ganadora consiste en retirar exactamente $r = n \pmod 4$ piedras, dejando al adversario en un múltiplo de 4.

3. **¿Por qué algunas posiciones tienen valor `-1` incluso si MAX todavía dispone de varios movimientos?**  
   Porque tener opciones legales de movimiento no garantiza que alguna conduzca a la victoria. En una posición perdedora ($n = 4, 8, \dots$), todas las transiciones disponibles conducen a estados desde los cuales el adversario dispone de una estrategia ganadora forzada si juega racionalmente.

4. **¿Puede haber más de una jugada igualmente buena?**  
   En posiciones perdedoras ($-1$), **todas las jugadas disponibles son igualmente perdedoras** frente a un rival perfecto. En ciertas variantes de juegos donde hay empates (como Tres en Raya) o en juegos con empates de utilidad, puede haber múltiples jugadas que garanticen el mismo valor máximo ($+1$ o $0$).

5. **¿Qué cambiaría si fuera obligatorio retirar únicamente `1` o `2` piedras?**  
   El ciclo modular se reduciría a período $2+1 = 3$. Las posiciones perdedoras pasarían a ser exactamente los **múltiplos de 3**: $\{3, 6, 9, 12, \dots\}$, y la jugada ganadora consistiría en retirar $n \pmod 3$ piedras.


## 9 Minimax con profundidad limitada

La siguiente versión admite:

- una profundidad máxima;
- una función de evaluación para estados no terminales.

Este esquema es mucho más cercano al utilizado en juegos reales.


In [10]:
def minimax_limitado(
    estado,
    profundidad,
    es_max,
    es_terminal,
    utilidad,
    sucesores,
    evaluar
):
    if es_terminal(estado):
        return utilidad(estado)

    if profundidad == 0:
        return evaluar(estado)

    valores = [
        minimax_limitado(
            hijo,
            profundidad - 1,
            not es_max,
            es_terminal,
            utilidad,
            sucesores,
            evaluar
        )
        for hijo in sucesores(estado)
    ]

    return max(valores) if es_max else min(valores)


# 10. Taller — Desarrollo y Solución Completa

---

### Implementación Completa de Minimax para Tres en Raya (Tic-Tac-Toe)

Representamos el tablero como una tupla de 9 posiciones indexadas del 0 al 8:
```text
 0 | 1 | 2
---+---+---
 3 | 4 | 5
---+---+---
 6 | 7 | 8
```
- `"X"` representa a MAX ($+1$).
- `"O"` representa a MIN ($-1$).
- `" "` representa una casilla vacía.
- Empate representa utilidad $0$.


In [11]:
def acciones(tablero):
    return [i for i, casilla in enumerate(tablero) if casilla == " "]

def resultado(tablero, accion, jugador):
    nuevo_tablero = list(tablero)
    nuevo_tablero[accion] = jugador
    return tuple(nuevo_tablero)

LINEAS_GANADORAS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8), # Filas
    (0, 3, 6), (1, 4, 7), (2, 5, 8), # Columnas
    (0, 4, 8), (2, 4, 6)             # Diagonales
]

def ganador(tablero):
    for a, b, c in LINEAS_GANADORAS:
        if tablero[a] != " " and tablero[a] == tablero[b] == tablero[c]:
            return tablero[a]
    return None

def terminal(tablero):
    if ganador(tablero) is not None:
        return True
    if " " not in tablero:
        return True
    return False

def utilidad(tablero):
    g = ganador(tablero)
    if g == "X":
        return 1
    elif g == "O":
        return -1
    else:
        return 0

def minimax_tictactoe(tablero, es_max):
    if terminal(tablero):
        return utilidad(tablero)

    jugador = "X" if es_max else "O"
    valores = [
        minimax_tictactoe(resultado(tablero, acc, jugador), not es_max)
        for acc in acciones(tablero)
    ]

    return max(valores) if es_max else min(valores)

def mejor_jugada_tictactoe(tablero, es_max):
    if terminal(tablero):
        return utilidad(tablero), None

    jugador = "X" if es_max else "O"
    opciones = []

    for acc in acciones(tablero):
        val = minimax_tictactoe(resultado(tablero, acc, jugador), not es_max)
        opciones.append((val, acc))

    if es_max:
        return max(opciones, key=lambda x: x[0])
    else:
        return min(opciones, key=lambda x: x[0])


### Comprobación de Tres en Raya en diversos tableros


In [12]:
def imprimir_tictactoe(t):
    for i in range(0, 9, 3):
        print(f" {t[i]} | {t[i+1]} | {t[i+2]} ")
        if i < 6:
            print("---+---+---")
    print()

# Tablero 1: Victoria inminente de MAX (X en fila superior)
tablero_victoria_max = (
    "X", "X", " ",
    "O", "O", " ",
    " ", " ", " "
)
print("Tablero 1 (Turno MAX - X debe ganar jugando en casilla 2):")
imprimir_tictactoe(tablero_victoria_max)
val1, jug1 = mejor_jugada_tictactoe(tablero_victoria_max, True)
print(f"Resultado -> Valor: {val1}, Mejor jugada en índice: {jug1}\n")

# Tablero 2: Bloqueo urgente (O amenaza con ganar)
tablero_bloqueo = (
    "O", "O", " ",
    "X", " ", " ",
    " ", " ", "X"
)
print("Tablero 2 (Turno MAX - X debe bloquear a O jugando en casilla 2):")
imprimir_tictactoe(tablero_bloqueo)
val2, jug2 = mejor_jugada_tictactoe(tablero_bloqueo, True)
print(f"Resultado -> Valor: {val2}, Mejor jugada en índice: {jug2}")


Tablero 1 (Turno MAX - X debe ganar jugando en casilla 2):
 X | X |   
---+---+---
 O | O |   
---+---+---
   |   |   

Resultado -> Valor: 1, Mejor jugada en índice: 2

Tablero 2 (Turno MAX - X debe bloquear a O jugando en casilla 2):
 O | O |   
---+---+---
 X |   |   
---+---+---
   |   | X 

Resultado -> Valor: 1, Mejor jugada en índice: 2


---

### Árbol de 3 niveles para Tres en Raya: Cálculo Manual vs. Python

Consideremos el siguiente estado a tres movimientos del final:
```text
Estado Raíz (Turno MAX - X):
 X | O | X
---+---+---
 O | X | O
---+---+---
   |   |   
```
Casillas vacías disponibles: $6, 7, 8$.

**Estructura del árbol:**
1. **Nivel 0 (Raíz - MAX):** MAX elige entre las casillas $6, 7$ u $8$.
2. **Nivel 1 (MIN):** Por cada elección de MAX, MIN juega en una de las 2 casillas restantes.
3. **Nivel 2 (MAX):** MAX juega en la última casilla disponible.
4. **Nivel 3 (Hojas Terminales):** Se evalúa la utilidad del juego.

**Cálculo Manual:**
- **Rama 1 ($MAX \to 6$):** Tablero con X en 6. Forma la diagonal $(2, 4, 6)$ $\implies$ ¡Victoria inmediata de X! Utilidad $= +1$. No requiere más niveles.
- **Rama 2 ($MAX \to 7$):**
  - Si $MIN \to 6$: MAX debe jugar en 8 $\implies$ Tablero lleno sin 3 en línea $\implies$ Empate (utilidad $0$).
  - Si $MIN \to 8$: MAX debe jugar en 6 $\implies$ Forma diagonal $(2, 4, 6)$ $\implies$ Victoria de X ($+1$).
  - MIN, siendo racional, elegirá $\min(0, 1) = 0$.
- **Rama 3 ($MAX \to 8$):**
  - Si $MIN \to 6$: MAX juega en 7 $\implies$ Empate (utilidad $0$).
  - Si $MIN \to 7$: MAX juega en 6 $\implies$ Diagonal de X $\implies$ Victoria ($+1$).
  - MIN elegirá $\min(0, 1) = 0$.

**Decisión en la Raíz:**
$$\max(\text{Rama 1: } +1, \text{Rama 2: } 0, \text{Rama 3: } 0) = +1$$
**Jugada elegida por MAX:** Casilla **6** (garantiza la victoria $+1$).

Comprobamos a continuación este resultado exacto con nuestra implementación en Python:


In [13]:
tablero_arbol = (
    "X", "O", "X",
    "O", "X", "O",
    " ", " ", " "
)

print("Tablero Raíz para el cálculo manual:")
imprimir_tictactoe(tablero_arbol)

val_arbol, jug_arbol = mejor_jugada_tictactoe(tablero_arbol, True)
print(f"Comprobación en Python: Valor Minimax = {val_arbol}, Jugada elegida = Casilla {jug_arbol}")
assert val_arbol == 1 and jug_arbol == 6, "Error en comprobación"
print("¡El cálculo manual coincide exactamente con el resultado de Python!")


Tablero Raíz para el cálculo manual:
 X | O | X 
---+---+---
 O | X | O 
---+---+---
   |   |   

Comprobación en Python: Valor Minimax = 1, Jugada elegida = Casilla 6
¡El cálculo manual coincide exactamente con el resultado de Python!


---

### Taller: Juego de las piedras modificado — Retirar únicamente {1, 2, 4} piedras

Modificamos las reglas:
- Cada jugador puede retirar únicamente `1`, `2` o `4` piedras.
- El jugador que retira la última piedra gana.

Implementamos y analizamos las posiciones ganadoras ($+1$), perdedoras ($-1$) y la mejor jugada para MAX de $1$ a $15$ piedras:


In [14]:
MOVIMIENTOS_MOD = (1, 2, 4)

def movs_validos_mod(piedras):
    return [m for m in MOVIMIENTOS_MOD if m <= piedras]

def minimax_piedras_mod(piedras, turno_max):
    if piedras == 0:
        return -1 if turno_max else 1

    valores = [
        minimax_piedras_mod(piedras - retirar, not turno_max)
        for retirar in movs_validos_mod(piedras)
    ]
    return max(valores) if turno_max else min(valores)

def mejor_mov_piedras_mod(piedras):
    opciones = []
    for retirar in movs_validos_mod(piedras):
        val = minimax_piedras_mod(piedras - retirar, False)
        opciones.append((val, retirar))
    
    mejor_val, mejor_mov = max(opciones, key=lambda x: x[0])
    return mejor_mov, mejor_val

print(f"{'Piedras':<8} | {'Valor':<8} | {'Estado':<12} | {'Mejor movimiento MAX'}")
print("-" * 52)
for p in range(1, 16):
    mov, val = mejor_mov_piedras_mod(p)
    estado_str = "Ganadora (+1)" if val == 1 else "Perdedora (-1)"
    mov_str = f"Retirar {mov}" if val == 1 else "Cualquiera (Pierde)"
    print(f"{p:<8} | {val:<8} | {estado_str:<12} | {mov_str}")


Piedras  | Valor    | Estado       | Mejor movimiento MAX
----------------------------------------------------
1        | 1        | Ganadora (+1) | Retirar 1
2        | 1        | Ganadora (+1) | Retirar 2
3        | -1       | Perdedora (-1) | Cualquiera (Pierde)
4        | 1        | Ganadora (+1) | Retirar 1
5        | 1        | Ganadora (+1) | Retirar 2
6        | -1       | Perdedora (-1) | Cualquiera (Pierde)
7        | 1        | Ganadora (+1) | Retirar 1
8        | 1        | Ganadora (+1) | Retirar 2
9        | -1       | Perdedora (-1) | Cualquiera (Pierde)
10       | 1        | Ganadora (+1) | Retirar 1
11       | 1        | Ganadora (+1) | Retirar 2
12       | -1       | Perdedora (-1) | Cualquiera (Pierde)
13       | 1        | Ganadora (+1) | Retirar 1
14       | 1        | Ganadora (+1) | Retirar 2
15       | -1       | Perdedora (-1) | Cualquiera (Pierde)


### Análisis de Posiciones Ganadoras y Perdedoras en el juego {1, 2, 4}

De acuerdo con la teoría matemática de los juegos de resta (*Subtraction Games / Sprague-Grundy*):
1. **Posiciones Perdedoras (-1):** Son exactamente los **múltiplos de 3**:
   $$\mathcal{P} = \{0, 3, 6, 9, 12, 15, \dots\} = \{n \in \mathbb{N} \mid n \equiv 0 \pmod 3\}$$
   - Si un jugador recibe un múltiplo de 3, cualquier jugada admisible ($1, 2$ o $4$) deja un residuo:
     - $n - 1 \equiv 2 \pmod 3$
     - $n - 2 \equiv 1 \pmod 3$
     - $n - 4 \equiv 2 \pmod 3$
     Ninguna jugada puede dejar al rival en otro múltiplo de 3.
2. **Posiciones Ganadoras (+1):** Son todas aquellas donde $n \not\equiv 0 \pmod 3$:
   - Si $n \equiv 1 \pmod 3$: MAX retira **1 piedra** (o 4 si $n \ge 4$), dejando al oponente en un múltiplo de 3.
   - Si $n \equiv 2 \pmod 3$: MAX retira **2 piedras**, dejando al oponente en un múltiplo de 3.

---

### Respuesta a la Pregunta final

> **¿Por qué una decisión que parece buena de manera inmediata puede ser mala después de considerar la respuesta del adversario?**

**Explicación Fundamental:**  
En problemas de búsqueda individual (como laberintos), una acción con alta ganancia inmediata (heurística greedy o miope) suele ser favorable. Sin embargo, en un **entorno adversarial**, el entorno no es pasivo: **reacciona activamente para destruir la posición del jugador**.

Una decisión que genera una recompensa inmediata (por ejemplo, capturar una pieza tentadora en ajedrez o avanzar un peón de forma agresiva) puede dejar al jugador en una posición con debilidades tácticas que el adversario explotará implacablemente en el siguiente turno (como una celada, un jaque mate forzado o una combinación letal).  
Minimax previene esta "miopía táctica" evaluando no la apariencia inmediata de la jugada, sino el **desenlace óptimo forzado tras la mejor respuesta posible del adversario**.


---

### Declaración de Uso de Inteligencia Artificial Generativa

- **Herramienta utilizada:** Asistente Inteligente Gemini Spark.
- **Propósito de uso:** Formulación estructurada de las funciones del juego de Tres en Raya, diseño de las pruebas de validación automatizadas y verificación del análisis matemático de los juegos de resta modular (juego de las piedras 1, 2, 4).
- **Partes de la actividad en las que fue empleada:** Implementación del motor de Minimax para Tic-Tac-Toe, verificación de árbol de decisión y redacción de las respuestas teóricas y conceptuales.
- **Aseguramiento de autoría:** Todos los algoritmos y cálculos fueron verificados, ejecutados y contrastados paso a paso por el estudiante para su presentación y defensa académica.
